# Baby Step 7 — Select Outside Counsel, Experts, and Vendors Using Conflict-Aware Gates

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

Baby Step 7 moves from strategy design to internal process architecture.

For each active matter, the notebook:

- creates synthetic outside-counsel profiles;
- creates synthetic expert profiles;
- creates synthetic litigation-vendor profiles;
- ranks fit against the accepted Baby Step 6 working case;
- applies hard conflict and independence gates;
- separates Wave 1, conditional Wave 2, reserve, and excluded candidates;
- records DEC-007;
- preserves Recommendation V1;
- prohibits any real instruction, engagement, or contact.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, statistics
from collections import defaultdict, Counter

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")
if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT/"00_System"/"Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))

if 6 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 6 is not complete.")

matters = json.loads((VAULT/"data"/"active_matters.json").read_text(encoding="utf-8"))
working_cases = json.loads((VAULT/"data"/"baby_step_6_working_cases.json").read_text(encoding="utf-8"))
recommendations = json.loads((VAULT/"data"/"baby_step_1_recommendations_v1.json").read_text(encoding="utf-8"))

print("Matters:",len(matters))
print("Working cases:",len(working_cases))


## Selection principle

A high fit score is not enough.

A candidate must also pass:

- conflict clearance;
- independence;
- required expertise;
- capacity;
- timing;
- budget compatibility;
- confidentiality capability;
- jurisdictional suitability.

Hard gates override weighted scores.


In [ ]:
RANKING_WEIGHTS = {
    "substantive_expertise":0.20,
    "procedural_expertise":0.15,
    "industry_expertise":0.10,
    "jurisdiction_fit":0.10,
    "working_case_fit":0.15,
    "capacity":0.10,
    "timing":0.08,
    "budget_fit":0.07,
    "relationship_quality":0.05
}
assert abs(sum(RANKING_WEIGHTS.values())-1.0)<1e-9

(VAULT/"00_System"/"Baby_Step_7_Selection_Model.json").write_text(
    json.dumps({
        "weights":RANKING_WEIGHTS,
        "hard_gates":[
            "hard_conflict",
            "independence_failure",
            "insufficient_capacity",
            "timing_incompatibility",
            "mandatory_expertise_failure"
        ],
        "purpose":"internal synthetic provider selection",
        "not_for":["real engagement","instruction","contact","procurement commitment"]
    },indent=2),encoding="utf-8"
)


## Synthetic provider universe

The notebook creates:

- 12 outside-counsel firms;
- 10 expert candidates;
- 8 litigation vendors.

All profiles are fictional.


In [ ]:
COUNSEL_SPECIALTIES = [
    "Corporate Governance",
    "M&A Disputes",
    "Technology Licensing",
    "Joint Venture Litigation",
    "Commercial Contracts",
    "Appellate Strategy"
]
EXPERT_SPECIALTIES = [
    "Forensic Accounting",
    "Valuation",
    "Software Systems",
    "Corporate Governance",
    "Supply Chain",
    "Damages Economics"
]
VENDOR_SPECIALTIES = [
    "E-Discovery",
    "Document Review",
    "Forensic Collection",
    "Trial Graphics",
    "Data Room",
    "Litigation Analytics"
]

counsel = []
for i in range(1,13):
    counsel.append({
        "provider_id":f"LAW-{i:02d}",
        "provider_type":"Outside Counsel",
        "name":f"Fictional Litigation Partners {i:02d}",
        "specialties":[COUNSEL_SPECIALTIES[(i-1)%len(COUNSEL_SPECIALTIES)],
                       COUNSEL_SPECIALTIES[i%len(COUNSEL_SPECIALTIES)]],
        "jurisdictions":["New York","Delaware","California","Texas"][(i-1)%4:(i-1)%4+1],
        "hourly_rate_index":60 + (i*3)%35,
        "capacity_score":55 + (i*7)%40,
        "timing_score":58 + (i*5)%38,
        "relationship_quality":50 + (i*6)%45,
        "conflict_state":["CLEAR","CONDITIONAL","HARD"][(i-1)%3],
        "independence_state":["INDEPENDENT","INDEPENDENT","REVIEW"][(i-1)%3],
        "synthetic":True
    })

experts = []
for i in range(1,11):
    experts.append({
        "provider_id":f"EXP-{i:02d}",
        "provider_type":"Expert",
        "name":f"Synthetic Expert {i:02d}",
        "specialty":EXPERT_SPECIALTIES[(i-1)%len(EXPERT_SPECIALTIES)],
        "jurisdictions":["National","New York","Delaware","California"][(i-1)%4],
        "fee_index":55 + (i*4)%40,
        "capacity_score":60 + (i*5)%35,
        "timing_score":62 + (i*6)%32,
        "independence_state":["INDEPENDENT","REVIEW","INDEPENDENT"][(i-1)%3],
        "conflict_state":["CLEAR","CONDITIONAL","CLEAR","HARD"][(i-1)%4],
        "synthetic":True
    })

vendors = []
for i in range(1,9):
    vendors.append({
        "provider_id":f"VEN-{i:02d}",
        "provider_type":"Vendor",
        "name":f"Synthetic Litigation Services {i:02d}",
        "specialty":VENDOR_SPECIALTIES[(i-1)%len(VENDOR_SPECIALTIES)],
        "fee_index":50 + (i*5)%38,
        "capacity_score":65 + (i*4)%30,
        "timing_score":60 + (i*7)%35,
        "confidentiality_score":70 + (i*3)%28,
        "conflict_state":["CLEAR","CLEAR","CONDITIONAL","HARD"][(i-1)%4],
        "synthetic":True
    })

provider_universe = counsel + experts + vendors
(VAULT/"data"/"baby_step_7_provider_universe.json").write_text(
    json.dumps(provider_universe,indent=2),encoding="utf-8"
)

print("Counsel:",len(counsel))
print("Experts:",len(experts))
print("Vendors:",len(vendors))


## Matter requirements

Each matter defines required provider capabilities derived from its Baby Step 6 working case.


In [ ]:
MATTER_REQUIREMENTS = {
    "MAT-001":{
        "counsel_specialty":"Corporate Governance",
        "expert_specialty":"Valuation",
        "vendor_specialty":"E-Discovery",
        "preferred_jurisdictions":["Delaware","New York"],
        "budget_ceiling":82,
        "minimum_capacity":65,
        "minimum_timing":68
    },
    "MAT-002":{
        "counsel_specialty":"M&A Disputes",
        "expert_specialty":"Forensic Accounting",
        "vendor_specialty":"Document Review",
        "preferred_jurisdictions":["New York","Delaware"],
        "budget_ceiling":78,
        "minimum_capacity":68,
        "minimum_timing":70
    },
    "MAT-003":{
        "counsel_specialty":"Technology Licensing",
        "expert_specialty":"Software Systems",
        "vendor_specialty":"Forensic Collection",
        "preferred_jurisdictions":["California","New York"],
        "budget_ceiling":85,
        "minimum_capacity":70,
        "minimum_timing":72
    },
    "MAT-004":{
        "counsel_specialty":"Joint Venture Litigation",
        "expert_specialty":"Corporate Governance",
        "vendor_specialty":"Litigation Analytics",
        "preferred_jurisdictions":["Delaware","New York"],
        "budget_ceiling":88,
        "minimum_capacity":65,
        "minimum_timing":66
    },
    "MAT-005":{
        "counsel_specialty":"Commercial Contracts",
        "expert_specialty":"Supply Chain",
        "vendor_specialty":"Trial Graphics",
        "preferred_jurisdictions":["Texas","New York"],
        "budget_ceiling":75,
        "minimum_capacity":66,
        "minimum_timing":70
    }
}

assert set(MATTER_REQUIREMENTS)=={m["matter_id"] for m in matters}


## Scoring and hard gates

The model ranks candidates but separately records hard-gate failures.


In [ ]:
def jurisdiction_fit(provider, req):
    values = provider.get("jurisdictions", [provider.get("jurisdiction","National")])
    if "National" in values:
        return 82
    return 100 if any(j in req["preferred_jurisdictions"] for j in values) else 55

def budget_fit(provider, req):
    index = provider.get("hourly_rate_index", provider.get("fee_index",70))
    if index <= req["budget_ceiling"]:
        return 100
    over = index - req["budget_ceiling"]
    return max(0,100-over*4)

def score_provider(provider, req, required_specialty, working_case):
    specialties = provider.get("specialties",[provider.get("specialty","")])
    substantive = 100 if required_specialty in specialties else 45
    procedural = 85 if provider["provider_type"]=="Outside Counsel" else 70
    industry = 80 if required_specialty in specialties else 55
    jfit = jurisdiction_fit(provider,req)
    working_fit = 90 if required_specialty.lower().split()[0] in working_case["selected_strategy"].lower() else 72
    capacity = provider["capacity_score"]
    timing = provider["timing_score"]
    budget = budget_fit(provider,req)
    relationship = provider.get("relationship_quality",65)

    components = {
        "substantive_expertise":substantive,
        "procedural_expertise":procedural,
        "industry_expertise":industry,
        "jurisdiction_fit":jfit,
        "working_case_fit":working_fit,
        "capacity":capacity,
        "timing":timing,
        "budget_fit":budget,
        "relationship_quality":relationship
    }
    weighted = {k:round(components[k]*RANKING_WEIGHTS[k],4) for k in RANKING_WEIGHTS}
    total = round(sum(weighted.values()),2)

    hard_failures = []
    if provider["conflict_state"]=="HARD":
        hard_failures.append("hard_conflict")
    if provider.get("independence_state")=="REVIEW" and provider["provider_type"]=="Expert":
        hard_failures.append("independence_failure")
    if capacity < req["minimum_capacity"]:
        hard_failures.append("insufficient_capacity")
    if timing < req["minimum_timing"]:
        hard_failures.append("timing_incompatibility")
    if substantive < 80:
        hard_failures.append("mandatory_expertise_failure")

    return {
        "provider_id":provider["provider_id"],
        "provider_type":provider["provider_type"],
        "name":provider["name"],
        "matter_id":working_case["matter_id"],
        "required_specialty":required_specialty,
        "components":components,
        "weighted_components":weighted,
        "selection_score":total,
        "conflict_state":provider["conflict_state"],
        "independence_state":provider.get("independence_state","N/A"),
        "hard_gate_failures":hard_failures,
        "eligible":len(hard_failures)==0
    }

selection_results = {}

for wc in working_cases:
    mid = wc["matter_id"]
    req = MATTER_REQUIREMENTS[mid]

    matter_results = {
        "counsel":[score_provider(p,req,req["counsel_specialty"],wc) for p in counsel],
        "experts":[score_provider(p,req,req["expert_specialty"],wc) for p in experts],
        "vendors":[score_provider(p,req,req["vendor_specialty"],wc) for p in vendors]
    }
    for category in matter_results:
        matter_results[category].sort(key=lambda x:(not x["eligible"],-x["selection_score"],x["provider_id"]))
    selection_results[mid]=matter_results

(VAULT/"data"/"baby_step_7_selection_results.json").write_text(
    json.dumps(selection_results,indent=2),encoding="utf-8"
)


## Wave structure

For each matter and provider category:

- Wave 1: eligible top candidate;
- Conditional Wave 2: strong but conflict or independence review required;
- Reserve: eligible lower-ranked candidate;
- Excluded: hard-gate failure.


In [ ]:
shortlists = []

for mid,groups in selection_results.items():
    for category,items in groups.items():
        eligible = [x for x in items if x["eligible"]]
        conditional = [
            x for x in items
            if not x["eligible"]
            and x["selection_score"]>=70
            and any(f in x["hard_gate_failures"] for f in ["independence_failure"])
        ]
        excluded = [x for x in items if not x["eligible"] and x not in conditional]

        wave1 = eligible[:1]
        reserve = eligible[1:3]
        wave2 = conditional[:2]

        for x in wave1:
            shortlists.append({**x,"category":category,"wave":"WAVE 1"})
        for x in wave2:
            shortlists.append({**x,"category":category,"wave":"CONDITIONAL WAVE 2"})
        for x in reserve:
            shortlists.append({**x,"category":category,"wave":"RESERVE"})
        for x in excluded:
            shortlists.append({**x,"category":category,"wave":"EXCLUDED"})

(VAULT/"data"/"baby_step_7_shortlists.json").write_text(
    json.dumps(shortlists,indent=2),encoding="utf-8"
)

wave_counts = Counter(x["wave"] for x in shortlists)
print(dict(wave_counts))


In [ ]:
def write_note(path,lines):
    path.write_text("\n".join(lines).strip()+"\n",encoding="utf-8")

provider_dir = VAULT/"19_Providers"
selection_dir = VAULT/"20_Selection_Reviews"
provider_dir.mkdir(parents=True,exist_ok=True)
selection_dir.mkdir(parents=True,exist_ok=True)

for p in provider_universe:
    lines = [
        "---",f"provider_id: {p['provider_id']}",
        f"provider_type: \"{p['provider_type']}\"",
        "synthetic: true","---","",
        f"# {p['provider_id']} — {p['name']}","",
        f"- Type: {p['provider_type']}",
        f"- Conflict state: {p['conflict_state']}",
        f"- Capacity: {p['capacity_score']}/100",
        f"- Timing: {p['timing_score']}/100","",
        "Synthetic profile for architecture testing only."
    ]
    write_note(provider_dir/f"{p['provider_id']}.md",lines)

for wc in working_cases:
    mid = wc["matter_id"]
    matter = next(m for m in matters if m["matter_id"]==mid)
    lines = [
        f"# {mid} — Provider Selection Review","",
        f"- Matter: [[../02_Active_Matters/{mid}]]",
        f"- Working case: [[../18_Strategy_Design/{wc['working_case_id']}]]","",
        "## Shortlist",""
    ]
    local = [x for x in shortlists if x["matter_id"]==mid and x["wave"]!="EXCLUDED"]
    for x in local:
        lines += [
            f"### {x['category']} — {x['wave']}","",
            f"- Provider: [[../19_Providers/{x['provider_id']}]]",
            f"- Score: {x['selection_score']}/100",
            f"- Conflict: {x['conflict_state']}",
            f"- Independence: {x['independence_state']}",
            f"- Eligible: {x['eligible']}",""
        ]
    lines += [
        "## Governance","",
        "This shortlist is internal only. No provider may be contacted or instructed."
    ]
    write_note(selection_dir/f"{mid}_Provider_Selection.md",lines)

print("Provider notes:",len(list(provider_dir.glob("*.md"))))
print("Selection reviews:",len(list(selection_dir.glob("*.md"))))


## Portfolio selection memorandum


In [ ]:
memo = [
    "# Baby Step 7 — Counsel, Expert, and Vendor Selection Memorandum","",
    "## Executive conclusion","",
    "Synthetic provider profiles were ranked against the five accepted working cases.",
    "Weighted fit scores were subordinated to conflict, independence, expertise, capacity, timing, and budget gates.","",
    "## Portfolio shortlist summary",""
]
for mid in sorted({x["matter_id"] for x in shortlists}):
    memo += [f"### {mid}",""]
    for category in ["counsel","experts","vendors"]:
        selected = [x for x in shortlists if x["matter_id"]==mid and x["category"]==category and x["wave"]=="WAVE 1"]
        if selected:
            x = selected[0]
            memo.append(f"- {category.title()}: [[../19_Providers/{x['provider_id']}]] — {x['selection_score']}/100")
        else:
            memo.append(f"- {category.title()}: No eligible Wave 1 candidate")
    memo.append("")

memo += [
    "## Governance conclusion","",
    "The shortlists support internal process planning only.",
    "No real engagement, instruction, contact, or procurement commitment is authorized."
]
write_note(VAULT/"10_Reports"/"Baby_Step_7_Provider_Selection_Memorandum.md",memo)


## Human decision — DEC-007

DEC-007 accepts the synthetic provider shortlists and conflict-aware selection architecture.

It authorizes:

- internal recipient verification design;
- disclosure-tier design;
- engagement-control design;
- simulated instruction workflow.

It does not authorize real contact or instruction.


In [ ]:
DECISION = {
    "decision_id":"DEC-007",
    "date":datetime.date.today().isoformat(),
    "title":"Accept Synthetic Provider Shortlists and Conflict Gates",
    "decision":"Accept the Baby Step 7 counsel, expert, and vendor shortlists as the internal process-design baseline.",
    "wave1_provider_ids":[x["provider_id"] for x in shortlists if x["wave"]=="WAVE 1"],
    "conditional_provider_ids":[x["provider_id"] for x in shortlists if x["wave"]=="CONDITIONAL WAVE 2"],
    "permitted_next_actions":[
        "internal recipient verification design",
        "disclosure-tier design",
        "engagement-control design",
        "simulated instruction workflow",
        "preserve Recommendation V1"
    ],
    "not_authorized":[
        "real provider contact",
        "provider instruction",
        "engagement letter",
        "procurement commitment",
        "Recommendation V2",
        "filing","service","party contact","court contact",
        "settlement offer","external legal advice","external distribution"
    ],
    "synthetic":True
}
(VAULT/"09_Decisions"/"DEC-007.json").write_text(json.dumps(DECISION,indent=2),encoding="utf-8")
lines = [
    "# DEC-007 — Accept Synthetic Provider Shortlists and Conflict Gates","",
    f"**Date:** {DECISION['date']}","","## Decision","",DECISION["decision"],"",
    "## Wave 1 providers",""
]
lines += [f"- [[../19_Providers/{pid}]]" for pid in DECISION["wave1_provider_ids"]]
lines += ["","## Conditional providers",""]
lines += [f"- [[../19_Providers/{pid}]]" for pid in DECISION["conditional_provider_ids"]] or ["- None"]
lines += ["","## Permitted next actions",""]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]
write_note(VAULT/"09_Decisions"/"DEC-007.md",lines)


In [ ]:
hot = [
    "# Current State — Hot Cache","",
    "## Recommendation state","",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.","",
    "## Provider-selection state","",
    f"- Wave 1 selections: {wave_counts.get('WAVE 1',0)}",
    f"- Conditional Wave 2: {wave_counts.get('CONDITIONAL WAVE 2',0)}",
    f"- Reserve: {wave_counts.get('RESERVE',0)}",
    f"- Excluded: {wave_counts.get('EXCLUDED',0)}","",
    "## Current decision","","- [[../09_Decisions/DEC-007]]","",
    "## Permitted","",
    "- Internal recipient verification design",
    "- Disclosure-tier design",
    "- Engagement-control design",
    "- Simulated instruction workflow","",
    "## Prohibited","",
    "- Real provider contact",
    "- Provider instruction",
    "- Engagement letters",
    "- Procurement commitments",
    "- Recommendation V2","",
    "## Next permitted experiment","",
    "Simulate controlled instruction and outreach without sending anything."
]
write_note(VAULT/"12_Hot_Cache"/"Current_State.md",hot)


In [ ]:
errors = []

provider_notes = list((VAULT/"19_Providers").glob("*.md"))
selection_notes = list((VAULT/"20_Selection_Reviews").glob("*.md"))
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))

if len(provider_notes)!=len(provider_universe):
    errors.append(f"Expected {len(provider_universe)} provider notes, found {len(provider_notes)}")
if len(selection_notes)!=5:
    errors.append(f"Expected 5 selection reviews, found {len(selection_notes)}")
if len(v1)!=5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")
if v2:
    errors.append("Recommendation V2 exists prematurely")

required = [
    VAULT/"data"/"baby_step_7_provider_universe.json",
    VAULT/"data"/"baby_step_7_selection_results.json",
    VAULT/"data"/"baby_step_7_shortlists.json",
    VAULT/"10_Reports"/"Baby_Step_7_Provider_Selection_Memorandum.md",
    VAULT/"09_Decisions"/"DEC-007.md",
    VAULT/"09_Decisions"/"DEC-007.json"
]
for p in required:
    if not p.exists():
        errors.append(f"Missing: {p}")

validation = {
    "validated_at":datetime.datetime.now().isoformat(),
    "provider_note_count":len(provider_notes),
    "selection_review_count":len(selection_notes),
    "wave_counts":dict(wave_counts),
    "recommendation_v1_count":len(v1),
    "recommendation_v2_count":len(v2),
    "decision":"DEC-007",
    "errors":errors,
    "passed":len(errors)==0
}
(VAULT/"11_Audit"/"Baby_Step_7_Validation.json").write_text(
    json.dumps(validation,indent=2),encoding="utf-8"
)
assert validation["passed"],errors
print(json.dumps(validation,indent=2))
print("BABY STEP 7 PASSED")


In [ ]:
state.update({
    "completed_steps":sorted(set(state.get("completed_steps",[])+[7])),
    "current_step":7,
    "next_step":8,
    "decision":"DEC-007",
    "provider_universe_count":len(provider_universe),
    "wave1_count":wave_counts.get("WAVE 1",0),
    "current_recommendation_version":1,
    "next_problem":"Simulate controlled instruction and outreach without sending anything.",
    "permission_state":{
        "observe":True,
        "organize":True,
        "browse":True,
        "internal_strategy_analysis":True,
        "evidence_governance":True,
        "committee_product":True,
        "controlled_internal_diligence":True,
        "remedies_and_damages_analysis":True,
        "counterparty_and_expert_selection":True,
        "simulated_instruction_design":True,
        "recommendation_v2":False,
        "external_action":False
    }
})
state_path.write_text(json.dumps(state,indent=2),encoding="utf-8")
audit = {
    "timestamp":datetime.datetime.now().isoformat(),
    "step":7,
    "action":"Selected synthetic outside counsel, experts, and vendors using conflict-aware gates.",
    "outputs":{
        "providers":len(provider_universe),
        "wave1":wave_counts.get("WAVE 1",0),
        "conditional_wave2":wave_counts.get("CONDITIONAL WAVE 2",0),
        "decision":"DEC-007"
    },
    "validation_passed":True
}
with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a",encoding="utf-8") as f:
    f.write(json.dumps(audit)+"\n")
